# Flight price prediction-REGRESSION

In [21]:
import pandas as pd
df = pd.read_csv(r"C:\Users\razih\OneDrive\Desktop\New\flight\Flight_Price.csv")
df.head(2)


,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,7h 25m,2 stops,No info,7662


In [22]:
df.isnull().sum()
df = df.bfill()


In [23]:

# splitting date of journey into day-month-year columns
df["day"] = pd.to_datetime(df["Date_of_Journey"], format="%d/%m/%Y").dt.day
df["month"] = pd.to_datetime(df["Date_of_Journey"], format="%d/%m/%Y").dt.month
df["year"] = pd.to_datetime(df["Date_of_Journey"], format="%d/%m/%Y").dt.year
df = df.drop(["Date_of_Journey"], axis=1)
df.head(2)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,day,month,year
0,IndiGo,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,24,3,2019
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,7h 25m,2 stops,No info,7662,1,5,2019


In [24]:

# changing duration to Hours and Minutes
df["Duration_H"] = df["Duration"].str.extract(r'(\d+)h').fillna(0).astype(int)
df["Duration_m"] = df["Duration"].str.extract(r'(\d+)m').fillna(0).astype(int)
df = df.drop(["Duration"], axis=1)
df.head(2)

,Airline,Source,Destination,Route,Dep_Time,Arrival_Time,Total_Stops,Additional_Info,Price,day,month,year,Duration_H,Duration_m
0,IndiGo,Banglore,New Delhi,BLR ? DEL,22:20,01:10 22 Mar,non-stop,No info,3897,24,3,2019,2,50
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,05:50,13:15,2 stops,No info,7662,1,5,2019,7,25


In [25]:

# Splitting Departure time into Hours and Minutes
df["Dep-H"] = df["Dep_Time"].str.split(":").str[0].astype(int)
df["Dep-m"] = df["Dep_Time"].str.split(":").str[1].astype(int)
df = df.drop(["Dep_Time"], axis=1)
df.head(2)

,Airline,Source,Destination,Route,Arrival_Time,Total_Stops,Additional_Info,Price,day,month,year,Duration_H,Duration_m,Dep-H,Dep-m
0,IndiGo,Banglore,New Delhi,BLR ? DEL,01:10 22 Mar,non-stop,No info,3897,24,3,2019,2,50,22,20
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,13:15,2 stops,No info,7662,1,5,2019,7,25,5,50


In [26]:

# changing arrival time into hours and minutes
df["Arrival_Time"] = df["Arrival_Time"].str.extract(r"(\d{1,2}:\d{2})")
df["Arrival_H"] = df["Arrival_Time"].str.split(":").str[0].astype(int)
df["Arrival_m"] = df["Arrival_Time"].str.split(":").str[1].astype(int)
df = df.drop(["Arrival_Time"], axis=1)
df.head(2)

,Airline,Source,Destination,Route,Total_Stops,Additional_Info,Price,day,month,year,Duration_H,Duration_m,Dep-H,Dep-m,Arrival_H,Arrival_m
0,IndiGo,Banglore,New Delhi,BLR ? DEL,non-stop,No info,3897,24,3,2019,2,50,22,20,1,10
1,Air India,Kolkata,Banglore,CCU ? IXR ? BBI ? BLR,2 stops,No info,7662,1,5,2019,7,25,5,50,13,15


In [27]:

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

# Separate features and target, then split BEFORE fitting any encoders
X = df.drop(["Price"], axis=1)
y = df["Price"]
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cols = ["Airline", "Source", "Destination", "Route", "Additional_Info"]

for col in cols:
    label = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    label.fit(x_train[[col]])   # fit ONLY on training data

    train_encoded = label.transform(x_train[[col]])
    test_encoded = label.transform(x_test[[col]])

    train_encoded_df = pd.DataFrame(train_encoded, columns=label.get_feature_names_out([col]), index=x_train.index)
    test_encoded_df = pd.DataFrame(test_encoded, columns=label.get_feature_names_out([col]), index=x_test.index)

    x_train = pd.concat([x_train, train_encoded_df], axis=1).drop(col, axis=1)
    x_test = pd.concat([x_test, test_encoded_df], axis=1).drop(col, axis=1)

# Label Encoding for ordinal feature - fit on train only
label = LabelEncoder()
x_train["Total_Stops"] = label.fit_transform(x_train["Total_Stops"])
x_test["Total_Stops"] = label.transform(x_test["Total_Stops"])


In [28]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"   # or however many cores you want to use

In [29]:
import numpy as np
from sklearn.metrics import mean_squared_error

train_rmse = np.sqrt(mean_squared_error(y_train, train_prediction))
test_rmse = np.sqrt(mean_squared_error(y_test, test_prediction))

In [30]:

# --- Model training ---
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

models = [
    LinearRegression(),
    DecisionTreeRegressor(),
    RandomForestRegressor(),
    GradientBoostingRegressor(),
    XGBRegressor(),
]

mlflow.set_experiment("Regression_Flight_price_prediction")
mlflow.set_tracking_uri("http://127.0.0.1:5000")

for model in models:
    with mlflow.start_run():
        model.fit(x_train, y_train)
        train_prediction = model.predict(x_train)
        test_prediction = model.predict(x_test)

        train_mse = mean_squared_error(y_train, train_prediction)
        train_r2 = r2_score(y_train, train_prediction)
        test_mse = mean_squared_error(y_test, test_prediction)
        test_r2 = r2_score(y_test, test_prediction)
        train_rmse = np.sqrt(train_mse)
        test_rmse = np.sqrt(test_mse)

        mlflow.log_param("model_name", type(model).__name__)
        mlflow.log_metric("train_mse", train_mse)
        mlflow.log_metric("train_rmse", train_rmse)
        mlflow.log_metric("train_r2", train_r2)
        mlflow.log_metric("test_mse", test_mse)
        mlflow.log_metric("test_rmse", test_rmse)
        mlflow.log_metric("test_r2", test_r2)
        mlflow.sklearn.log_model(model, type(model).__name__)

        print(f"\n{type(model).__name__}")
        print("*Train*")
        print(f"MSE: {train_mse}  RMSE: {train_rmse}")
        print(f"R2: {train_r2}")
        print("\n*Test*")
        print(f"MSE: {test_mse}  RMSE: {test_rmse}")
        print(f"R2: {test_r2}")

2026/09/09 17:06:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



LinearRegression
*Train*
MSE: 5287539.763570019  RMSE: 2299.4651037947974
R2: 0.7516020698509418

*Test*
MSE: 5395654.5413103085  RMSE: 2322.8548257070024
R2: 0.7450546590787882
🏃 View run secretive-asp-698 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/7b1ca62bb64240518939b57000931905
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/09/09 17:06:20 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



DecisionTreeRegressor
*Train*
MSE: 94385.73389109915  RMSE: 307.2226129227781
R2: 0.9955659490079524

*Test*
MSE: 3449229.400301565  RMSE: 1857.210112050213
R2: 0.8370234864662407
🏃 View run lyrical-gnu-222 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/aa919481a70f4e85914b508eb9fe9bc6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/09/09 17:06:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



RandomForestRegressor
*Train*
MSE: 346723.7501455204  RMSE: 588.8325314939048
R2: 0.9837116190665739

*Test*
MSE: 2758802.486520015  RMSE: 1660.9643242767181
R2: 0.8696462430878074
🏃 View run unruly-crane-652 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/8ca5db7edf12469085e3050635a2e38c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/09/09 17:06:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



GradientBoostingRegressor
*Train*
MSE: 3632491.737916844  RMSE: 1905.9096877650954
R2: 0.829352880672639

*Test*
MSE: 3819143.356626854  RMSE: 1954.2628678422086
R2: 0.8195450065181966
🏃 View run carefree-shrike-455 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/d616f1fdca004124bb45b26006ca454d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516


2026/09/09 17:06:59 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



XGBRegressor
*Train*
MSE: 1107120.875  RMSE: 1052.1981158508127
R2: 0.9479897022247314

*Test*
MSE: 2004553.375  RMSE: 1415.8225082968556
R2: 0.905284583568573
🏃 View run wistful-pig-856 at: http://127.0.0.1:5000/#/experiments/458478889590703516/runs/1cb765e41c584f70a3b61103628e6666
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/458478889590703516
